# Day 2: Feature Analysis & Model Evaluation
## MAIA — AI Song Attribution Assessment

This notebook:
1. Runs the full MAIA pipeline on all sample pairs
2. Visualizes per-component scores
3. Evaluates attribution accuracy on labeled pairs
4. Shows the `compare_tracks()` API in action

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import json

from pipeline import compare_tracks

sns.set_theme(style='darkgrid')
print('Imports OK')

## 1. Single Pair Demo

In [ ]:
# Example usage:
# result = compare_tracks(
#     'data/samples/original_track.mp3',
#     'data/samples/ai_derivative.mp3',
#     verbose=True
# )
# print(json.dumps(result, indent=2))
print('Provide track paths above to run attribution scoring')

## 2. Batch Evaluation on Multiple Pairs

In [ ]:
def eval_pair_list(pairs: list) -> pd.DataFrame:
    """
    Evaluate a list of (track_a, track_b, true_label) tuples.
    true_label: 1 = attributed (AI cover of A), 0 = unrelated
    """
    results = []
    for track_a, track_b, true_label in pairs:
        print(f'Processing: {Path(track_a).name} vs {Path(track_b).name}')
        result = compare_tracks(track_a, track_b)
        result['track_a'] = Path(track_a).name
        result['track_b'] = Path(track_b).name
        result['true_label'] = true_label
        result['correct'] = (result['attribution_score'] >= 0.5) == bool(true_label)
        results.append(result)
    
    df = pd.DataFrame(results)
    print(f'\nAccuracy: {df["correct"].mean():.2%}')
    print(f'Mean score (positives): {df[df["true_label"]==1]["attribution_score"].mean():.4f}')
    print(f'Mean score (negatives): {df[df["true_label"]==0]["attribution_score"].mean():.4f}')
    return df

# Example:
# pairs = [
#     ('data/samples/song1_orig.mp3', 'data/samples/song1_suno.mp3', 1),
#     ('data/samples/song1_orig.mp3', 'data/samples/unrelated.mp3', 0),
#     ('data/samples/song2_orig.mp3', 'data/samples/song2_udio.mp3', 1),
# ]
# df = eval_pair_list(pairs)
print('eval_pair_list() defined — add labeled pairs to run evaluation')

## 3. Score Component Visualization

In [ ]:
def plot_score_radar(result: dict, title: str = 'MAIA Attribution Scores'):
    """Radar chart showing all 4 score components."""
    categories = ['Semantic\nSimilarity', 'Melodic\nAlignment', 
                  'Structural\nMatch', 'AI Artifact\nScore']
    values = [
        result['semantic_similarity'],
        result['melodic_alignment'],
        result['structural_correspondence'],
        result['ai_artifact_score'],
    ]
    
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    values_plot = values + values[:1]
    angles += angles[:1]
    
    fig, (ax_radar, ax_bar) = plt.subplots(1, 2, figsize=(14, 6),
                                            subplot_kw=dict(polar=False))
    
    # Bar chart version (cleaner)
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    bars = ax_bar.bar(categories, values, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
    ax_bar.set_ylim(0, 1)
    ax_bar.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Attribution threshold (0.5)')
    ax_bar.set_ylabel('Score')
    ax_bar.set_title(f'{title}\nFinal Score: {result["attribution_score"]:.4f}')
    
    for bar, val in zip(bars, values):
        ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax_bar.legend()
    
    # Verdict box
    color = '#4CAF50' if result['attribution_score'] >= 0.5 else '#F44336'
    ax_radar.text(0.5, 0.5, result['verdict'],
                  ha='center', va='center', fontsize=11,
                  transform=ax_radar.transAxes,
                  bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3))
    ax_radar.axis('off')
    
    plt.tight_layout()
    plt.show()

# Example usage:
# plot_score_radar(result, title='Song X vs AI Cover')
print('plot_score_radar() defined')

## 4. Threshold Tuning
Find the optimal decision threshold using labeled pairs.

In [ ]:
def find_optimal_threshold(df: pd.DataFrame):
    """Plot ROC-style curve and find best threshold for attribution detection."""
    from sklearn.metrics import roc_curve, auc, precision_recall_curve
    
    y_true = df['true_label'].values
    y_score = df['attribution_score'].values
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC Curve
    axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
    axes[0].plot([0, 1], [0, 1], color='navy', linestyle='--')
    axes[0].set_xlim([0, 1])
    axes[0].set_ylim([0, 1.05])
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve — Attribution Detection')
    axes[0].legend()
    
    # Score distributions
    axes[1].hist(df[df['true_label']==1]['attribution_score'], bins=20, alpha=0.7,
                 label='Attributed (AI cover)', color='green')
    axes[1].hist(df[df['true_label']==0]['attribution_score'], bins=20, alpha=0.7,
                 label='Unrelated', color='red')
    axes[1].set_xlabel('Attribution Score')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Score Distributions')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Youden's J optimal threshold
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    print(f'Optimal threshold (Youden J): {thresholds[best_idx]:.4f}')
    print(f'At this threshold: TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f}')
    return thresholds[best_idx]

print('find_optimal_threshold() defined')